In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import sys
import wandb
import time
import json
import random
import math
import pickle
import pandas as pd
import numpy as np
import re

import transformers
from transformers import Trainer, AutoTokenizer, AutoModelForCausalLM, Mamba2ForCausalLM, TrainingArguments, DataCollatorWithPadding, Mamba2Config, GenerationMixin, DataCollatorForMultipleChoice, DataCollatorForLanguageModeling
from transformers.utils import add_start_docstrings, add_start_docstrings_to_model_forward, logging, replace_return_docstrings, add_code_sample_docstrings

from typing import Dict, List, Optional, Tuple, Union
from dataclasses import dataclass
from datasets import load_dataset, Dataset, DatasetDict, load_from_disk
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef
from sklearn.model_selection import train_test_split

import evaluate

from peft import PeftModel, LoraConfig, get_peft_model, TaskType
import peft.tuners.lora.layer as pl

import torch
from torch import nn
import torch.nn.functional as F
from torch.nn.modules.loss import CrossEntropyLoss
from torch.nn import Parameter, ParameterList

import pennylane as qml

from trl import SFTTrainer

from tqdm import tqdm
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence

from tqdm.notebook import tqdm
from IPython.display import clear_output

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [ ]:
SEED = 1755
NUM_EPOCHS = 1
BATCH_SIZE = 1
LEARNING_RATE = 5e-4
LORA_R = 2
NUM_QLAYERS = 4

set_seed(SEED)

MODEL_NAME="AntonV/mamba2-130m-hf"
OUTPUT_DIR = "mamba2_cp/07_APC"+"_RANK"+str(LORA_R)
output_file = os.path.join(
    r"output", "07_APC"+"_RANK"+str(LORA_R)+"_EP"+str(NUM_EPOCHS)+"_SEED"+str(SEED)+".json"
)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir="mamba2_model")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="cuda",
    dtype=torch.float32,
    cache_dir="mamba2_model"
)

print(model)

In [ ]:
df_dataset = pd.read_csv("../iot_datasets/iot_resource_allocation_dataset.csv")
def row_to_json_prompt(r):
    payload = {
        "Timestamp":               r["Timestamp"],
        "Device_ID":               r["Device_ID"],
        "Temp":                    r["Sensor_Data"].split(",")[0].split(":")[1].strip(),
        "Humidity":                r["Sensor_Data"].split(",")[1].split(":")[1].strip(),
        "Workload_Type":           r["Workload_Type"],
        "Processing_Tier":         r["Processing_Tier"],
        "CPU_Usage(%)":            r["CPU_Usage(%)"],
        "Memory_Usage(MB)":        r["Memory_Usage(MB)"],
        "Network_Latency(ms)":     r["Network_Latency(ms)"],
        "Network_Jitter(ms)":      r["Jitter(ms)"],
        "Task_Execution_Time(ms)": r["Task_Execution_Time(ms)"],
        "Predicted_Resource_Allocation(%)": r["Predicted_Resource_Allocation(%)"],
    }
    return f"{json.dumps(payload, ensure_ascii=False)}\n###\nPredict the Actual Resource Allocation(%)"

df_dataset["text"]  = df_dataset.apply(row_to_json_prompt, axis=1)
df_dataset["label"] = df_dataset["Actual_Resource_Allocation(%)"].astype(str)
df = df_dataset[["text", "label"]]

print(df["text"][0])
print()
print("Actual Resource Allocation(%):")
print(df["label"][0])

In [ ]:
CACHE_PATH = "../iot_datasets/iot_resource_allocation_hf_dataset"

if os.path.exists(CACHE_PATH):
    ds_dict = load_from_disk(CACHE_PATH)
else:
    train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, shuffle=False)

    ds_dict = DatasetDict({
        "train": Dataset.from_pandas(train_df.reset_index(drop=True)),
        "test":  Dataset.from_pandas(test_df.reset_index(drop=True))
    })

    ds_dict.save_to_disk(CACHE_PATH)
    print(f"✓ {CACHE_PATH}")

In [ ]:
iot_train_hf_dataset = Dataset.from_list(ds_dict["train"])
iot_test_hf_dataset = Dataset.from_list(ds_dict["test"])
print(iot_train_hf_dataset[0])

In [ ]:
def length_only(example):
    ids = tokenizer(example["text"], add_special_tokens=True,
                    padding=False, truncation=False).input_ids
    return {"seq_len": len(ids)}

train_with_len = ds_dict["train"].map(length_only)
MAX_LEN = int(np.max(train_with_len["seq_len"]))
print("Raw longest length:", MAX_LEN)

MAX_LEN = min(MAX_LEN, tokenizer.model_max_length)
print("Effective MAX_LEN:", MAX_LEN)

In [ ]:
def tok_fn(batch):
    prompt_ids = tokenizer(batch["text"], truncation=True, max_length=MAX_LEN).input_ids
    answer_ids = tokenizer(batch["label"], add_special_tokens=False).input_ids
    input_ids  = [p + answer_ids[i] + [tokenizer.eos_token_id] for i, p in enumerate(prompt_ids)]
    labels     = [[-100]*len(p) + answer_ids[i] + [tokenizer.eos_token_id] for i, p in enumerate(prompt_ids)]
    attn_mask  = [[1]*len(ids) for ids in input_ids]
    return {"input_ids": input_ids, "attention_mask": attn_mask, "labels": labels}

tokd = ds_dict.map(
    tok_fn,
    batched=True,
    remove_columns=["text", "label"]
)

tokd.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

In [ ]:
def collate_fn(features):
    input_ids_batch = []
    attention_mask_batch = []
    labels_batch = []

    for f in features:
        ids = f["input_ids"]
        if isinstance(ids, torch.Tensor):
            ids = ids.detach().clone()
        else:
            ids = torch.tensor(ids, dtype=torch.long)
        input_ids_batch.append(ids)

        mask = f["attention_mask"]
        if isinstance(mask, torch.Tensor):
            mask = mask.detach().clone()
        else:
            mask = torch.tensor(mask, dtype=torch.long)
        attention_mask_batch.append(mask)

        lab = f["labels"]
        if isinstance(lab, torch.Tensor):
            lab = lab.detach().clone()
        else:
            lab = torch.tensor(lab, dtype=torch.long)
        labels_batch.append(lab)

    input_ids = pad_sequence(
        input_ids_batch, batch_first=True, padding_value=tokenizer.pad_token_id
    )
    attention_mask = pad_sequence(
        attention_mask_batch, batch_first=True, padding_value=0
    )
    labels = pad_sequence(
        labels_batch, batch_first=True, padding_value=-100
    )

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }

In [ ]:
def deep_block(weights, wires):
    L = len(weights) // 6
    w_resh = qml.math.reshape(weights, (L, 2, 3))
    qml.StronglyEntanglingLayers(w_resh, wires=wires)

def build_custom_pairs(num_qubits: int, rank: int = 2):
    stage_A = [(2*i, 2*i + 1) for i in range(num_qubits // 2)]
    stage_B = [
        (i, i + gap)
        for gap in range(1, num_qubits)
        for i   in range(num_qubits - gap)
    ]

    root_wires = [(num_qubits - 1 - i) % num_qubits for i in range(rank)][::-1]

    return [stage_A, stage_B], root_wires

def custom_ttn(weights, wires, pair_layers):
    k = 0
    for layer in pair_layers:
        for a, b in layer:
            deep_block(weights[k], [wires[a], wires[b]])
            k += 1

class AmplitudeMPSQuantum(nn.Module):
    def __init__(self, in_features, out_features, rank, num_layers):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.rank = rank

        self.num_qubits = math.ceil(math.log2(self.in_features))
        
        if self.num_qubits < self.rank:
            self.num_qubits = self.rank
            
        self.embedding_dim = 2 ** self.num_qubits
        
        self.n_layers = num_layers
        self.rotations_per_wire = 3
        self.n_params_block = self.n_layers * 2 * self.rotations_per_wire

        self.pair_layers, self.root_wires = build_custom_pairs(self.num_qubits, self.rank)
        self.n_blocks = sum(len(l) for l in self.pair_layers)

        self.q_params = nn.Parameter(torch.randn(self.n_blocks, self.n_params_block)*0.01 )
        n_blocks = self.n_blocks
        n_params_block = self.n_params_block

        self.dev = qml.device("default.qubit", wires=self.num_qubits)
        
        @qml.qnode(self.dev, interface="torch", diff_method="backprop")
        def circuit(params, data_amp):
            qml.StatePrep(data_amp, wires=range(self.num_qubits), normalize=False, validate_norm=False)
            
            custom_ttn(params, list(range(self.num_qubits)), self.pair_layers)
            
            return [qml.expval(qml.PauliZ(w)) for w in self.root_wires]
        
        self.circuit = circuit

    def reset_parameters(self):
        nn.init.xavier_uniform_(self.q_params)  # 使用 Xavier 初始化

    def _pad_and_normalize(self, x_1d: torch.Tensor) -> torch.Tensor:
        pad_len = self.embedding_dim - x_1d.shape[0]
        padded = F.pad(x_1d, (0, pad_len), mode='constant', value=0.0)
        eps = torch.finfo(padded.dtype).eps
        padded_norm = F.normalize(padded, p=2.0, dim=0, eps=eps)

        return padded_norm

    def forward(self, x_1d: torch.Tensor) -> torch.Tensor:
        padded = self._pad_and_normalize(x_1d)
        
        qc_probs = self.circuit(self.q_params, padded)
        if isinstance(qc_probs, (tuple, list)):
            qc_outs = torch.stack(qc_probs, dim=-1)
        else:
            qc_outs = qc_probs

        return qc_outs.to(x_1d.device)
    
class QuantumLoRALayer(nn.Module):
    def __init__(self, base_module: nn.Linear, r=2, lora_alpha=8.0, num_qubits=6, num_layers=4):
        super().__init__()
        
        self.base_module = base_module
        self.lora_alpha = lora_alpha

        self.out_features = base_module.out_features
        self.in_features = base_module.in_features

        self.rank = r
        self.lora_alpha = lora_alpha
        self.num_qubits = num_qubits
        self.num_layers = num_layers

        self.scaling = self.lora_alpha / self.rank

        self.qc = AmplitudeMPSQuantum(self.in_features, self.out_features, self.rank, self.num_layers)
        self.projectB  = nn.Linear(self.rank, self.out_features, bias=False)
    
    @property
    def weight(self):
        return self.base_module.weight

    @property
    def bias(self):
        return self.base_module.bias

    def forward(self, x):        
        target_dev = x.device

        if self.projectB.weight.device != target_dev:
            self.projectB = self.projectB.to(target_dev)
        
        y_base = self.base_module(x.to(target_dev))

        orig_shape = x.shape
        if x.dim() == 2:
            batch_size = orig_shape[0]
            seq_len = 1
            x_2d = x.view(batch_size, seq_len, self.in_features)
        elif x.dim() == 3:
            batch_size, seq_len, _ = orig_shape
            x_2d = x.view(batch_size, seq_len, self.in_features)
        else:
            raise ValueError("x must be 2D or 3D with last dim=in_features.")
        
        batch_size, seq_length, feature_dim = x_2d.shape

        saved_seq_len = x_2d.clone()
            
        qc_vmapped = torch.vmap(
                self.qc,
        )

        quantum_input = x_2d.view(-1, feature_dim).to(dtype=torch.float32, device=target_dev)
        
        yz_tensor = qc_vmapped(quantum_input)

        quantum_outs_tensor=yz_tensor.view(batch_size, seq_length, self.rank).to(dtype=x.dtype, device=target_dev)

        lora_2b = self.projectB(quantum_outs_tensor)


        if seq_len == 1:
            lora_res = lora_2b.view(batch_size, self.out_features)
        else:
            lora_res = lora_2b.view(batch_size, seq_len, self.out_features)

        y = y_base + lora_res

        if x.dim() == 2 and y.dim() == 3:
            y = y.squeeze(1)
        elif x.dim() == 3 and y.dim() == 2:
            y = y.unsqueeze(1)

        return y.to(target_dev)
         
def replace_with_quantum_lora(model: nn.Module, rank, lora_alpha, num_layers, target_modules, parent_name=""):
    for child_name, child_module in model.named_children():
        full_name = f"{parent_name}.{child_name}" if parent_name else child_name

        if isinstance(child_module, nn.Linear) and any(t in full_name for t in target_modules):
            qlora = QuantumLoRALayer(
                base_module = child_module,
                r           = rank,
                lora_alpha  = lora_alpha,
                num_layers  = num_layers,
            )
            setattr(model, child_name, qlora)
        else:
            replace_with_quantum_lora(
                child_module, rank, lora_alpha, num_layers,
                target_modules, parent_name=full_name
            )

In [ ]:
replace_with_quantum_lora(
    model=model,
    rank=LORA_R,
    lora_alpha=LORA_R*2,
    num_layers=NUM_QLAYERS,
    target_modules=["in_proj", "out_proj"],
)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_EPOCHS,
    save_strategy="no",
    logging_steps=100,
    fp16=False,
    bf16=False,
    remove_unused_columns=False,
    label_names=["labels"],
)

trainer = Trainer(
    model=model,
    processing_class=tokenizer,
    args=training_args,
    train_dataset=tokd["train"],
    data_collator=collate_fn,
)

In [ ]:
def freeze_model_parameters(model: nn.Module):
    for name, param in model.named_parameters():
        param.requires_grad = False

    for module in model.modules():
        if isinstance(module, QuantumLoRALayer):
            if hasattr(module, 'qc'):
                module.qc.reset_parameters()
                for p in module.qc.parameters():
                    p.requires_grad = True
                    
            module.projectB.reset_parameters()
            for p in module.projectB.parameters():
                p.requires_grad = True

freeze_model_parameters(trainer.model)

trainable = [n for n, p in model.named_parameters() if p.requires_grad]
print("Trainable parameters:\n", "\n".join(trainable))

In [ ]:
def count_params(module, trainable_only=True):
    return sum(p.numel() for p in module.parameters()
               if (p.requires_grad if trainable_only else True))

def fmt(n): return f"{n:,}"

def pct(val, total): 
    return f"({100 * val / total:.3f}%)" if total > 0 else "(0.000%)"

def _numel_trainable_qparams(qparams):
    if isinstance(qparams, Parameter):
        return qparams.numel() if qparams.requires_grad else 0
    if isinstance(qparams, ParameterList):
        return sum(p.numel() for p in qparams if p.requires_grad)
    try:
        return sum(p.numel() for p in qparams if getattr(p, "requires_grad", False))
    except Exception:
        return 0

def report_param_counts(model: nn.Module, return_dict=True):
    total_params = sum(p.numel() for p in model.parameters())
    trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    stats = {}

    str_global = f"total={fmt(total_params)}, trainable={fmt(trainable)} {pct(trainable, total_params)}, frozen={fmt(total_params-trainable)}"
    print(f"[GLOBAL] {str_global}")
    stats["param_global"] = str_global

    linear_trainable_all = 0
    for name, m in model.named_modules():
        if isinstance(m, nn.Linear):
            linear_trainable_all += count_params(m, True)
    str_lin_all = f"{fmt(linear_trainable_all)} {pct(linear_trainable_all, total_params)}"
    print(f"[Linear ALL] trainable params = {str_lin_all}")
    stats["param_linear_all"] = str_lin_all

    linear_trainable_excl_qc = 0
    for name, m in model.named_modules():
        if isinstance(m, nn.Linear) and ".qc." not in name and not name.endswith(".qc"):
            linear_trainable_excl_qc += count_params(m, True)
    str_lin_excl_qc = f"{fmt(linear_trainable_excl_qc)} {pct(linear_trainable_excl_qc, total_params)}"
    print(f"[Linear EXCL qc] trainable params = {str_lin_excl_qc}")
    stats["param_linear_excl_qc"] = str_lin_excl_qc

    qc_gate_params = 0
    for _, m in model.named_modules():
        if isinstance(m, AmplitudeMPSQuantum) and hasattr(m, "q_params"):
            qc_gate_params += _numel_trainable_qparams(m.q_params)
    str_qc_gates = f"{fmt(qc_gate_params)} {pct(qc_gate_params, total_params)}"
    print(f"[QuantumCircuit] trainable gate params (q_params) = {str_qc_gates}")
    stats["param_qc_gates"] = str_qc_gates

    qc_lin_params = 0
    for _, m in model.named_modules():
        if isinstance(m, AmplitudeMPSQuantum) and hasattr(m, "lin_layers"):
            qc_lin_params += count_params(m.lin_layers, True)
    str_qc_lin = f"{fmt(qc_lin_params)} {pct(qc_lin_params, total_params)}"
    print(f"[QuantumCircuit] trainable linear params (lin_layers) = {str_qc_lin}")
    stats["param_qc_linear"] = str_qc_lin

    for name, m in model.named_modules():
        if isinstance(m, QuantumLoRALayer):
            qc = _numel_trainable_qparams(m.qc.q_params) if hasattr(m.qc, "q_params") else 0
            qlin = count_params(m.qc.lin_layers, True) if hasattr(m.qc, "lin_layers") else 0
            pb = count_params(m.projectB, True) if hasattr(m, "projectB") else 0
            ap = count_params(m.projectA, True) if hasattr(m, "projectA") else 0
            print(f"[{name}] qc_gates={fmt(qc)}, qc_lin_layers={fmt(qlin)}, "
                  f"projectA={fmt(ap)}, projectB={fmt(pb)}, sum={fmt(qc+qlin+ap+pb)}")

    approx = linear_trainable_excl_qc + qc_gate_params + qc_lin_params
    print(f"[CHECK] non_qc_linear + qc_gates + qc_lin_layers = {fmt(approx)} "
          f"({'OK' if approx==trainable else 'NOTE: differs'})")
          
    if return_dict:
        return stats

report_param_counts(trainer.model)
print(trainer.model)

In [ ]:
torch.cuda.empty_cache()

In [ ]:
train_result=trainer.train()

train_loss = train_result.training_loss
train_runtime_sec = train_result.metrics["train_runtime"]

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

In [ ]:
def run_mamba(model, context):

    text = f"{context}"
    input_ids = tokenizer.encode(text, return_tensors="pt").cuda()
    attention_mask = torch.ones_like(input_ids).cuda()

    out = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_length=input_ids.shape[1]+10,
        eos_token_id=tokenizer.eos_token_id,
    )

    decoded = tokenizer.batch_decode(out)[0]
    cleaned = decoded.replace(text, "")
    cleaned = cleaned.replace("<|endoftext|>", "")
    cleaned = cleaned.split("\n\n")[0].strip()
    lines = cleaned.splitlines()
    if lines:
        cleaned = lines[0].strip()

    return cleaned

In [ ]:
context="{\"Timestamp\": \"2024-03-25 12:00:00\", \"Device_ID\": \"D14\", \"Temp\": \"32°C\", \"Humidity\": \"72%\", \"Workload_Type\": \"Data Analytics\", \"Processing_Tier\": \"Device\", \"CPU_Usage(%)\": 24, \"Memory_Usage(MB)\": 2276, \"Network_Latency(ms)\": 29, \"Network_Jitter(ms)\": 8, \"Task_Execution_Time(ms)\": 189, \"Predicted_Resource_Allocation(%)\": 54}\n###\nPredict the Actual Resource Allocation(%)"
print(run_mamba(trainer.model, context=context))

In [ ]:
N_SHOW = 10 

all_data = iot_test_hf_dataset
total_qs = len(all_data)

results = []
num_correct = 0

squared_error_sum = 0.0    
absolute_error_sum = 0.0   
valid_pred_count = 0       

start_time_total = time.time()

pbar = tqdm(total=total_qs, desc=f"Processing Examples: 0/{total_qs}", dynamic_ncols=True)

for i, data in enumerate(all_data, start=1):
    start_time = time.time()
    
    text = data["text"]
    y_true = float(data["label"])

    guess = run_mamba(trainer.model, text)
    elapsed_time = time.time() - start_time

    try:
        pred_value = float(re.findall(r"[-+]?\d*\.\d+|\d+", guess)[-1])
    except (IndexError, ValueError):
        pred_value = np.nan

    if not np.isnan(pred_value):
        error = pred_value - y_true
        squared_error_sum += error ** 2
        absolute_error_sum += abs(error)
        valid_pred_count += 1

        is_correct = abs(error) <= 3
    else:
        is_correct = False 

    if is_correct:
        num_correct += 1

    if i <= N_SHOW:
        log_msg = (
            f"Question {i}/{total_qs}\n"
            f"Context: {text}\n"
            f"y_true: {y_true}\n"
            f"pred: {pred_value}\n"
            f"{'✅ Correct' if is_correct else '❌ Incorrect'}\n"
            f"Time taken: {elapsed_time:.2f}s\n"
            + "=" * 80
        )
        tqdm.write(log_msg)

    results.append({
        "idx": i - 1,
        "question": text,
        "answer": y_true,
        "guess": pred_value,
        "is_correct": is_correct,
        "time": elapsed_time,
    })

    pbar.update(1)
    current_acc = (num_correct / i) * 100
    pbar.set_description(f"Processing Examples: {i}/{total_qs} | Acc: {current_acc:.2f}%")

pbar.close()

accuracy = num_correct / total_qs * 100
total_runtime = time.time() - start_time_total
avg_inference_time_per_sample = total_runtime / total_qs if total_qs > 0 else 0.0

if valid_pred_count > 0:
    rmse = (squared_error_sum / valid_pred_count) ** 0.5
    mae = absolute_error_sum / valid_pred_count
else:
    rmse = float("nan")
    mae = float("nan")

inf_h, inf_rem = divmod(total_runtime, 3600)
inf_m, inf_s = divmod(inf_rem, 60)
inference_time_str = f"{int(inf_h)}h{int(inf_m)}m{inf_s:.2f}s"

try:
    h, rem = divmod(train_runtime_sec, 3600)
    m, s = divmod(rem, 60)
    train_time_str = f"{int(h)}h{int(m)}m{s:.2f}s"
    print(f"\nTraining completed in {train_time_str} with loss: {train_loss:.4f}")
except NameError:
    train_loss = "N/A"
    train_time_str = "N/A"

print(f"Number of correct answers: {num_correct}/{total_qs}")
print(f"Accuracy rate: {accuracy:.2f}%")
print(f"RMSE: {rmse:.4f}, MAE: {mae:.4f}")
print(f"Total inference runtime: {inference_time_str}")
print(f"Average inference time per sample: {avg_inference_time_per_sample:.4f}s")

param_stats_dict = report_param_counts(trainer.model, return_dict=True)

final_output = {
    "summary": {
        "num_correct": num_correct,
        "total_questions": total_qs,
        "accuracy": accuracy,
        "accuracy_str": f"{accuracy:.2f}%",
        "rmse": round(rmse, 4) if not np.isnan(rmse) else None,
        "mae": round(mae, 4) if not np.isnan(mae) else None,
        "valid_predictions": valid_pred_count,

        **param_stats_dict,
        
        "train_loss": round(train_loss, 4) if isinstance(train_loss, float) else train_loss,
        "train_time": train_time_str,
        
        "inference_runtime_sec": f"{total_runtime:.2f}s",
        "inference_time": inference_time_str,
        "avg_inference_time_per_sample_sec": round(avg_inference_time_per_sample, 4)
    },
    "results": results  
}

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(final_output, f, indent=2, ensure_ascii=False)

print(f"Detailed results and summary saved in {output_file}")